# Full Training Comparison: CheXNet vs Novel Pipeline

This notebook compares the results of the full dataset training for both models:
1. **Baseline CheXNet**: `chexnet_best.pth` (DenseNet121)
2. **Novel Pipeline**: `novel_full_best.pth` (Hybrid CNN + Swin Transformer + Attention)

The comparison is performed on the validation set of the full 50% dataset subset.

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

TARGET_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 
    'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
DATA_DIR = 'data/images'
METADATA_PATH = 'data/Data_Entry_2017_v2020.csv'

## 1. System Architectures

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        res = self.conv(torch.cat([avg_out, max_out], dim=1))
        return x * self.sigmoid(res)

class HybridCheXNet(nn.Module):
    def __init__(self):
        super().__init__()
        cnn = models.densenet121(weights=None)
        self.cnn_features = cnn.features
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0)
        self.spatial_att = SpatialAttention()
        self.classifier = nn.Sequential(nn.Linear(1024 + 768, 512), nn.ReLU())
        self.final_fc = nn.Linear(512, 15)

    def forward(self, x):
        c_feat = self.spatial_att(self.cnn_features(x))
        c_feat = self.cnn_pool(c_feat).view(x.size(0), -1) 
        v_feat = self.vit(x)
        return self.final_fc(self.classifier(torch.cat([c_feat, v_feat], dim=1)))

def load_models():
    # Baseline
    baseline = models.densenet121(weights=None)
    baseline.classifier = nn.Linear(baseline.classifier.in_features, 15)
    if os.path.exists('chexnet_best.pth'):
        baseline.load_state_dict(torch.load('chexnet_best.pth', map_location=device))
        print("Loaded Baseline weights.")
    
    # Novel
    novel = HybridCheXNet()
    if os.path.exists('novel_full_best.pth'):
        novel.load_state_dict(torch.load('novel_full_best.pth', map_location=device))
        print("Loaded Novel Pipeline weights.")
    
    return baseline.to(device).eval(), novel.to(device).eval()

## 2. Dataset Preparation

In [ ]:
class ComparisonDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None, use_clahe=False):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.use_clahe = use_clahe
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image Index']
            img_path = os.path.join(self.root_dir, img_name)
            
            if self.use_clahe:
                image = cv2.imread(img_path)
                if image is None: return torch.zeros((3, 224, 224)), torch.zeros(15)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
                l, a, b = cv2.split(lab)
                cl = self.clahe.apply(l)
                image = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)
                image = Image.fromarray(image)
            else:
                image = Image.open(img_path).convert('RGB')
                
            label = torch.tensor(self.dataframe.iloc[idx]['label_vec'], dtype=torch.float32)
            if self.transform: image = self.transform(image)
            return image, label
        except:
            return torch.zeros((3, 224, 224)), torch.zeros(15)

def get_loaders():
    df = pd.read_csv(METADATA_PATH)
    def encode(l): return [1 if c in str(l).split('|') else 0 for c in TARGET_CLASSES]
    df['label_vec'] = df['Finding Labels'].apply(encode)
    
    available = set(os.listdir(DATA_DIR))
    df = df[df['Image Index'].isin(available)].reset_index(drop=True)
    
    _, val_df = train_test_split(df, test_size=0.1, random_state=42)
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    base_loader = DataLoader(ComparisonDataset(val_df, DATA_DIR, transform, use_clahe=False), batch_size=32, shuffle=False)
    novel_loader = DataLoader(ComparisonDataset(val_df, DATA_DIR, transform, use_clahe=True), batch_size=32, shuffle=False)
    
    return base_loader, novel_loader

## 3. Evaluation Engine

In [ ]:
def get_aucs(model, loader):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            out = torch.sigmoid(model(imgs.to(device)))
            all_preds.append(out.cpu().numpy())
            all_labels.append(labels.numpy())
    
    preds = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    
    aucs = []
    for i in range(len(TARGET_CLASSES)):
        try:
            aucs.append(roc_auc_score(labels[:, i], preds[:, i]))
        except:
            aucs.append(0.5)
    return aucs

## 4. Run Comparison

In [ ]:
print("Loading models...")
baseline, novel = load_models()

print("Preparing data loaders...")
base_loader, novel_loader = get_loaders()

print("\nBenchmarking Baseline CheXNet...")
base_aucs = get_aucs(baseline, base_loader)

print("\nBenchmarking Novel Pipeline...")
novel_aucs = get_aucs(novel, novel_loader)

results = []
for i, cls in enumerate(TARGET_CLASSES):
    better = "Novel Hybrid" if novel_aucs[i] > base_aucs[i] else "Baseline CheXNet"
    diff = novel_aucs[i] - base_aucs[i]
    results.append({
        "Pathology": cls,
        "Baseline AUC": base_aucs[i],
        "Novel AUC": novel_aucs[i],
        "Improvement": diff,
        "Winner": better
    })

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("FULL TRAINING LABEL-BY-LABEL COMPARISON")
print("="*60)
display(results_df)

print("\nDETAILED WINNER REPORT:")
print("-"*40)
for index, row in results_df.iterrows():
    print(f"{row['Pathology']:<20}: {row['Winner']} is better ({row['Improvement']:+.4f} AUC shift)")

total_novel_wins = (results_df['Winner'] == 'Novel Hybrid').sum()
total_labels = len(results_df)
print(f"\nFINAL VERDICT: Novel Pipeline outperforms Baseline on {total_novel_wins}/{total_labels} pathologies.")

print(f"\nMean Baseline AUC: {np.mean(base_aucs):.4f}")
print(f"Mean Novel AUC: {np.mean(novel_aucs):.4f}")
print(f"Overall Improvement: {np.mean(novel_aucs) - np.mean(base_aucs):.4f}")

## 5. Visualization

In [ ]:
plt.figure(figsize=(15, 8))
melted_df = results_df.melt(id_vars="Pathology", value_vars=["Baseline AUC", "Novel AUC"], 
                            var_name="Model", value_name="AUC")

sns.barplot(data=melted_df, x="Pathology", y="AUC", hue="Model", palette="viridis")
plt.xticks(rotation=45)
plt.title("Full Training Comparison: Pathology-wise AUC")
plt.ylim(0.5, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()